In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name
#username and password are set for testing purposes like this in the init.
# if this were a real application the username and password would be set through a ui element that would pop up on the users screen
username = ""
password = ""

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

rescue_criteria = {
    'Water': {
        'breeds': ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        'sex': "Intact Female",
        'age_min': 26,
        'age_max': 156
    },
    'Mountain': {
        'breeds': ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"],
        'sex': "Intact Male",
        'age_min': 26,
        'age_max': 156
    },
    'Disaster': {
        'breeds': ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"],
        'sex': "Intact Male",
        'age_min': 20,
        'age_max': 300
    }
}
## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso_Salvare_Logo.png' # replace with your own image
if os.path.exists(image_filename):
    encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
    print("✅ Logo loaded successfully!")
else:
    encoded_image = ""
    print("❌ Logo file NOT found! Check the filename and folder.")

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Center([
        # Grazioso Salvare Logo with required URL link to www.snhu.edu
        html.A(
            html.Img(src=f'data:image/png;base64,{encoded_image}', height=140),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.H1('Grazioso Salvare Search and Rescue Dashboard', 
                style={'margin-top': '10px', 'margin-bottom': '5px'}),
        html.H4('Created by: Malaki', 
                style={'color': '#555', 'margin-bottom': '20px'})
    ]),
    html.Hr(),
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '30px', 'font-size': '18px'}
        )
    ], style={'text-align': 'center', 'padding': '15px', 'backgroundColor': '#f8f9fa'}),

    html.Hr(),

    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                         editable=False,
                         filter_action="native",
                         sort_action="native",
                         page_action="native",
                         page_current=0,
                         page_size=10,
                         row_selectable="single",
                         selected_rows=[],
                         style_table={'overflowX': 'auto'},
                         style_cell={'textAlign': 'left', 'padding': '8px'},
                         style_header={'backgroundColor': 'rgb(230, 230, 230)', 'fontWeight': 'bold'}
                         ),
                     
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################




@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
    if filter_type == 'Reset':
        filtered_df = pd.DataFrame.from_records(db.read({}))
    else:
        crit = rescue_criteria[filter_type]
        query = {
            "animal_type": "Dog",
            "breed": {"$in": crit['breeds']},
            "sex_upon_outcome": crit['sex'],
            "age_upon_outcome_in_weeks": {"$gte": crit['age_min'], "$lte": crit['age_max']}
        }
        filtered_df = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if viewData is None or len(viewData) == 0:
        return dcc.Graph(figure=px.pie(title="No data available"))
    
    dff = pd.DataFrame.from_dict(viewData)
    fig = px.pie(
        dff,
        names='outcome_type',
        title='Distribution of Animal Outcomes',
        hole=0.4
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')
    return dcc.Graph(figure=fig)
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index): 
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='inline', port=8051, debug=False)

Read operation returned 10000 document(s).
✅ Logo loaded successfully!


 * Running on http://127.0.0.1:8051/ (Press CTRL+C to quit)
127.0.0.1 - - [19/Apr/2026 02:54:56] "GET /_alive_a2a1ab1e-8ceb-4779-8a56-d6fcc5853ad1 HTTP/1.1" 200 -


127.0.0.1 - - [19/Apr/2026 02:54:56] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "POST /_dash-update-component HTTP/1.1" 500 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Apr/2026 02:54:57] "POST /_dash-update-component HTTP/1.1" 200 -


---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

Read operation returned 10000 document(s).


127.0.0.1 - - [19/Apr/2026 02:54:58] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Apr/2026 02:54:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:54:58] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Apr/2026 02:55:00] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:55:00] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:55:00] "POST /_dash-update-component HTTP/1.1" 500 -


...

Read operation returned 5 document(s).
---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
IndexError: list index out of range



127.0.0.1 - - [19/Apr/2026 02:56:59] "POST /_dash-update-component HTTP/1.1" 500 -
127.0.0.1 - - [19/Apr/2026 02:56:59] "POST /_dash-update-component HTTP/1.1" 200 -


Read operation returned 10000 document(s).


127.0.0.1 - - [19/Apr/2026 02:57:00] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:57:02] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2026 02:57:03] "POST /_dash-update-component HTTP/1.1" 500 -


---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
IndexError: list index out of range

